# 1. Carga de datos y preparación

In [5]:
import pandas as pd
import joblib
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, root_mean_squared_error

# 1. Cargar el dataset limpio
# Mantenemos la misma estructura de rutas que usaste en el EDA
file_path = Path('../data/dataset_ecuador_limpio.csv') 
df = pd.read_csv(file_path)

# 2. Definir las variables de entrada (X) y la variable objetivo (y)
# Usamos estrictamente las que pide la rúbrica
features = ['provincia', 'lugar', 'num_dormitorios', 'num_banos', 'area', 'num_garages']
target = 'precio'

X = df[features]
y = df[target]

# 3. Dividir en datos de entrenamiento (80%) y prueba (20%)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Datos de entrenamiento: {X_train.shape[0]} filas")
print(f"Datos de prueba: {X_test.shape[0]} filas")

Datos de entrenamiento: 400 filas
Datos de prueba: 100 filas


# 2. Construcción del Pipeline y Entrenamiento

In [6]:
# 1. Definir qué columnas son categóricas
categorical_features = ['provincia', 'lugar']

# 2. Crear el transformador para las variables categóricas
# Usamos OneHotEncoder y le decimos que ignore categorías nuevas (handle_unknown='ignore')
# Esto es vital para la API, por si alguien ingresa un "Lugar" que el modelo no vio en el entrenamiento
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_features)
    ],
    remainder='passthrough' # Las variables numéricas pasan tal cual
)

# 3. Ensamblar el Pipeline con el Modelo
# Elegimos Random Forest (en la siguiente celda de Markdown pondremos la justificación)
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1))
])

# 4. Entrenar el modelo (¡Toda la magia ocurre en esta línea!)
pipeline.fit(X_train, y_train)
print("¡Modelo entrenado exitosamente!")

¡Modelo entrenado exitosamente!


# 3. Evaluación y Exportación

In [7]:
# 1. Hacer predicciones con los datos de prueba
y_pred = pipeline.predict(X_test)

# 2. Calcular métricas de evaluación
mae = mean_absolute_error(y_test, y_pred)
rmse = root_mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("--- Evaluación del Modelo ---")
print(f"Error Absoluto Medio (MAE): ${mae:.2f}")
print(f"Raíz del Error Cuadrático Medio (RMSE): ${rmse:.2f}")
print(f"Coeficiente de Determinación (R²): {r2:.4f}")

# 3. Exportar el Pipeline completo
# Esto guarda las transformaciones y el modelo en un solo archivo .joblib
modelo_path = Path('../modelo/modelo_alquileres.joblib')
joblib.dump(pipeline, modelo_path)

print(f"\n¡Modelo serializado y guardado en: {modelo_path}!")

--- Evaluación del Modelo ---
Error Absoluto Medio (MAE): $271.24
Raíz del Error Cuadrático Medio (RMSE): $529.60
Coeficiente de Determinación (R²): 0.6611

¡Modelo serializado y guardado en: ../modelo/modelo_alquileres.joblib!
